# Nemotron-3-Nano LoRA SFT with CoT-Selected Training Data (Unsloth)

This notebook is the **Unsloth** variant — same data pipeline and training config, but using Unsloth for ~2x faster training and lower VRAM.

In [ ]:
import kagglehub

BASE_MODEL_NAME = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"Base model: {BASE_MODEL_NAME}")

# Training config (matched to 0.72 standard notebook winning settings)
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MAX_SEQ_LEN = 4096
NUM_EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 1e-4
SEED = 123

## Setup & Model Loading

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages unsloth trl peft transformers datasets accelerate bitsandbytes
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import torch
from unsloth import FastLanguageModel

MODEL_PATH = BASE_MODEL_NAME
print(f"Model path: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    dtype=None,
)
print("Model loaded.")

In [ ]:
target_modules = ['in_proj', 'out_proj', 'up_proj', 'down_proj']

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()

## Training

In [ ]:
import pandas as pd
import re
import time
from datasets import Dataset as HFDataset
from trl import SFTTrainer
from transformers import TrainingArguments

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# --- Dataset path ---
DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection/train_split_with_cot.csv"
df = pd.read_csv(DATASET_PATH)
print(f"Full dataset: {len(df)} rows")
print(df["type"].value_counts().sort_index())

# --- Type-based sampling (matched to 0.72 standard notebook) ---
TYPE_SAMPLES = {
    "Numeral Conversion": 300,
    "Gravitational Constant": 400,
    "Unit Conversion": 700,
    "Text Encryption": 700,
    "Bit Manipulation": 607,        # all available
    "Equation Transformation": 200,  # all available
}

sampled_dfs = []
for ptype, n_samples in TYPE_SAMPLES.items():
    subset = df[df["type"] == ptype]
    if n_samples >= len(subset):
        sampled = subset
    else:
        sampled = subset.sample(n=n_samples, random_state=SEED)
    print(f"  {ptype}: {len(subset)} -> {len(sampled)}")
    sampled_dfs.append(sampled)

train_df = pd.concat(sampled_dfs, ignore_index=True)
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"\nTraining samples: {len(train_df)}")

# --- Build SFT dataset ---
records = []
for _, row in train_df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot = str(row["generated_cot"])
    if not cot or cot == "nan" or len(cot.strip()) < 5:
        continue
    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
    user_content = prompt + PROMPT_SUFFIX
    assistant_content = cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
    records.append({"messages": [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})

dataset = HFDataset.from_list(records)
print(f"SFT records: {len(records)}")

# --- Pre-tokenize with chat template (Unsloth pattern) ---
dataset = dataset.map(lambda ex: {
    "text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
})

# --- Training ---
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=TrainingArguments(
        output_dir="/kaggle/working/sft_output",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        optim="adamw_8bit",
        seed=SEED,
        report_to="none",
        dataloader_num_workers=2,
        eval_strategy="no",
    ),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    dataset_kwargs={"skip_prepare_dataset": False},
)

print("Starting SFT training...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"Training done in {elapsed/60:.1f} min")

# Save adapter
ADAPTER_DIR = "/kaggle/working/sft_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "chat_template.jinja",
]

src_adapter_dir = "/kaggle/working/sft_adapter"
print("Packaging freshly trained adapter from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")